# Do reaction starting materials / candidates correlate with the product fingerprint?

**Motivation.** MIST's reaction-conditioning pathway (`--aux-dim`,
`--reaction-metadata-file`) feeds the model a mean-pooled Morgan fingerprint
of a compound's known reaction starting materials (and, separately, a
`candidates` source of algorithmically-generated decoy structures) as extra
context alongside the spectrum. Before investing in a fancier architecture
for using this signal (e.g. treating it as a residual/warm-start rather than
just extra context concatenated onto the pooled spectrum representation), we
want to know: **does this signal actually correlate with the true product
fingerprint at all**, or is it closer to noise?

This notebook replicates the *exact* production featurization
(`src/mist/data/aux_featurizers.py`'s `SmilesSetFeaturizer`: mean-pooled
`morgan4096` fingerprints, same leakage guard as
`datasets.attach_reactions` for `candidates`) and measures Tanimoto
similarity between the pooled aux fingerprint and the real product
fingerprint, compared against a random-pairing baseline.

In [ ]:
import sys

sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rdkit import RDLogger

RDLogger.DisableLog("rdApp.*")  # silence per-molecule parse warnings (real-world
# SMILES data triggers many harmless ones -- e.g. unusual bond types, explicit
# Hs without neighbors -- that would otherwise dominate this notebook's output

from mist.data import data
from mist.data.aux_featurizers import SmilesSetFeaturizer

FP_NAME = "morgan4096"
SEED = 0

NIST23_LABELS = "/orcd/data/ccoley/001/msms_data/nist23/labels.tsv"
REACTION_FILES = [
    "../data/nist23/reaction_metadata_uspto.tsv",
    "../data/nist23/reaction_metadata_cas.tsv",
    "../data/nist23/reaction_metadata_pistachio.tsv",
]

rng = np.random.default_rng(SEED)

## Helper functions

In [ ]:
def tanimoto(a: np.ndarray, b: np.ndarray) -> float:
    """Generalized Jaccard/Tanimoto similarity between two (possibly
    non-binary, since the aux FP is mean-pooled over >1 molecule) fingerprint
    vectors. Reduces to standard Tanimoto when both inputs are binary."""
    a = a.astype(np.float64)
    b = b.astype(np.float64)
    minsum = np.minimum(a, b).sum()
    maxsum = np.maximum(a, b).sum()
    if maxsum == 0:
        return np.nan
    return minsum / maxsum


def load_reaction_metadata() -> pd.DataFrame:
    dfs = [
        pd.read_csv(f, sep="\t", dtype=str, keep_default_na=False)
        for f in REACTION_FILES
    ]
    return pd.concat(dfs, ignore_index=True)


def load_nist23_products() -> pd.DataFrame:
    """One row per unique compound (inchikey) in NIST23 -- labels.tsv is one
    row per spectrum/acquisition, so many rows share the same compound."""
    labels = pd.read_csv(NIST23_LABELS, sep="\t")
    products = labels[["inchikey", "smiles"]].drop_duplicates(subset="inchikey")
    return products.set_index("inchikey")


def compute_product_fps(products: pd.DataFrame) -> dict:
    """Morgan FP for every unique NIST23 compound's own SMILES -- the ground
    truth MIST is trained to predict."""
    featurizer = SmilesSetFeaturizer(fp_names=[FP_NAME])
    out = {}
    for inchikey, row in products.iterrows():
        mol = data.Mol.MolFromSmiles(row["smiles"])
        if mol is None:
            continue
        out[inchikey] = featurizer._fp_featurizer._featurize(mol)
    return out


def compute_aux_fps(
    reaction_df: pd.DataFrame, column: str, exclude_own_inchikey: bool = True
) -> dict:
    """Mean-pooled Morgan FP per matched inchikey across all its reaction
    rows' `column` entries -- replicates aux_featurizers.SmilesSetFeaturizer
    .featurize exactly. Leakage guard (exclude_own_inchikey) mirrors
    datasets.attach_reactions: filters candidates matching the product's own
    InChIKey, not SMILES string equality."""
    featurizer = SmilesSetFeaturizer(fp_names=[FP_NAME])
    smiles_by_inchikey: dict = {}
    for inchikey, group in reaction_df.groupby("inchikey"):
        all_smiles = []
        for val in group[column]:
            if not val:
                continue
            all_smiles.extend(s for s in val.split(";") if s)
        smiles_by_inchikey.setdefault(inchikey, []).extend(all_smiles)

    out = {}
    for inchikey, smiles_list in smiles_by_inchikey.items():
        if exclude_own_inchikey:
            filtered = []
            for smiles in smiles_list:
                mol = data.Mol.MolFromSmiles(smiles, inchikey=None)
                if mol is not None and mol.get_inchikey() == inchikey:
                    continue
                filtered.append(smiles)
            smiles_list = filtered
        fp = featurizer.featurize(smiles_list)
        if fp.sum() > 0:
            out[inchikey] = fp
    return out


def analyze(product_fps: dict, aux_fps: dict, label: str):
    common = sorted(set(product_fps) & set(aux_fps))
    print(f"=== {label} ===")
    print(f"compounds with both a product FP and a non-empty aux FP: {len(common)}")
    if not common:
        return None

    real_sims = np.array([tanimoto(product_fps[k], aux_fps[k]) for k in common])

    shuffled_products = [product_fps[k] for k in common]
    perm = rng.permutation(len(shuffled_products))
    random_sims = np.array(
        [
            tanimoto(shuffled_products[perm[i]], aux_fps[common[i]])
            for i in range(len(common))
        ]
    )

    real_sims = real_sims[~np.isnan(real_sims)]
    random_sims = random_sims[~np.isnan(random_sims)]

    print(
        f"REAL   pairing: mean={real_sims.mean():.4f}  median={np.median(real_sims):.4f}  "
        f"std={real_sims.std():.4f}  n={len(real_sims)}"
    )
    print(
        f"RANDOM pairing: mean={random_sims.mean():.4f}  median={np.median(random_sims):.4f}  "
        f"std={random_sims.std():.4f}  n={len(random_sims)}"
    )
    print(f"Difference (real - random) in mean: {real_sims.mean() - random_sims.mean():.4f}")
    print()

    return {"label": label, "common": common, "real_sims": real_sims, "random_sims": random_sims}

## Load data

In [ ]:
print("Loading NIST23 products...")
products = load_nist23_products()
print(f"{len(products)} unique compounds in NIST23")

print("Computing product Morgan FPs...")
product_fps = compute_product_fps(products)
print(f"{len(product_fps)} parseable")

print("Loading reaction metadata (USPTO + CAS + Pistachio)...")
reaction_df = load_reaction_metadata()
print(f"{len(reaction_df)} reaction-metadata rows")

## Analysis 1: starting materials vs. product

In [ ]:
print("Computing starting-materials mean-pooled FPs (production logic)...")
sm_fps = compute_aux_fps(reaction_df, "starting_materials", exclude_own_inchikey=False)
sm_result = analyze(product_fps, sm_fps, "starting_materials vs. product")

## Analysis 2: candidates vs. product

In [ ]:
print("Computing candidates mean-pooled FPs (production logic + leakage guard)...")
cand_fps = compute_aux_fps(reaction_df, "candidates", exclude_own_inchikey=True)
cand_result = analyze(product_fps, cand_fps, "candidates vs. product")

## Distributions: real pairing vs. random baseline

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)

for ax, result in zip(axes, [sm_result, cand_result]):
    bins = np.linspace(0, 1, 41)
    ax.hist(result["random_sims"], bins=bins, alpha=0.55, label="random pairing", color="#999999")
    ax.hist(result["real_sims"], bins=bins, alpha=0.65, label="real pairing", color="#1f77b4")
    ax.axvline(result["real_sims"].mean(), color="#1f77b4", linestyle="--", linewidth=1.5)
    ax.axvline(result["random_sims"].mean(), color="#666666", linestyle="--", linewidth=1.5)
    ax.set_title(result["label"] + f"\n(n={len(result['real_sims'])})")
    ax.set_xlabel("Tanimoto similarity to product")
    ax.legend()

axes[0].set_ylabel("count")
fig.suptitle("Aux-signal Tanimoto similarity to true product fingerprint: real vs. random pairing")
fig.tight_layout()
fig.savefig("../results/figures/reaction_aux_fp_correlation.png", dpi=150)
plt.show()

## Summary table

In [ ]:
summary = pd.DataFrame(
    [
        {
            "aux_source": r["label"],
            "n": len(r["real_sims"]),
            "real_mean": r["real_sims"].mean(),
            "real_median": np.median(r["real_sims"]),
            "random_mean": r["random_sims"].mean(),
            "random_median": np.median(r["random_sims"]),
            "diff_mean": r["real_sims"].mean() - r["random_sims"].mean(),
        }
        for r in [sm_result, cand_result]
    ]
)
summary

## Coverage: how much of NIST23 does each source actually cover?

In [ ]:
n_total = len(product_fps)
for r in [sm_result, cand_result]:
    n = len(r["common"])
    print(f"{r['label']:35s}: {n:6d} / {n_total} compounds covered ({100 * n / n_total:.1f}%)")

## Takeaways

- Both `starting_materials` and `candidates` show a **real, substantial**
  correlation with the true product fingerprint -- roughly **4x higher**
  mean Tanimoto similarity than random pairing (~0.31 and ~0.34 vs. ~0.07-0.09),
  well outside the noise (std ~0.14-0.18).
- `candidates` correlates *slightly more strongly* than `starting_materials`,
  despite being algorithmically-generated decoys rather than the real
  causal chemistry -- plausibly because `candidates` are specifically
  constructed as structurally-plausible near-products (small edits/
  combinations), while starting materials can differ from the product by a
  full synthetic transformation.
- **Coverage is partial**: `starting_materials` covers roughly half of
  NIST23's unique compounds; `candidates` covers a much smaller fraction.
  Any benefit from this signal is capped by coverage unless extended.

**Implication**: this signal is strong enough to justify further
architecture investment (e.g. treating the aux fingerprint as a residual/
warm-start rather than just extra context), not just noise the model has to
learn to ignore.